In [25]:
import pandas as pd
import os
import numpy as np
import pysam

In [26]:
def add_feature(existing_features, new_feature):
    return existing_features + (new_feature,)

In [27]:
from typing import List, Optional
from Bio import Phylo

def tip_order_from_nexus_biopython(
    nexus_path: str,
    tree_label: Optional[str] = None
) -> List[str]:
    """
    Returns the left-to-right tip (sample) order from a NEXUS file using Biopython.
    If `tree_label` is given, picks that named tree; else the first tree in the file.
    """
    trees = list(Phylo.parse(nexus_path, "nexus"))
    if not trees:
        raise ValueError("No trees found in NEXUS file.")

    if tree_label is not None:
        matches = [t for t in trees if (t.name or "") == tree_label]
        if not matches:
            raise ValueError(f"Tree labeled '{tree_label}' not found.")
        tree = matches[0]
    else:
        tree = trees[0]

    tip_order = [clade.name for clade in tree.find_clades(order="preorder") if clade.is_terminal()]
    return tip_order

In [28]:
phylogeneticOrder=()
tips_bio = tip_order_from_nexus_biopython("/LeeLab/HPRC/chromosomeY/Information/phylogeny/142males_HGSVC_HPRC_CEPH_241125_150M-FOR_SHARING.nex")         # first tree

for sample in tips_bio:
    phylogeneticOrder = add_feature(phylogeneticOrder, sample)
   
print(phylogeneticOrder)
print(len(phylogeneticOrder))

('HG02984', 'HG01890', 'HG02647', 'HG02666', 'HG02668', 'HG03225', 'NA19043', 'NA19384', 'HG005', 'NA18952', 'NA18983', 'HG02572', 'HG03248', 'HG03098', 'HG03050', 'NA19239', 'HG01074', 'HG01109', 'HG01106', 'NA19331', 'HG01252', 'HG01457', 'HG03065', 'HG02717', 'HG03471', 'HG02011', 'HG03371', 'HG02486', 'HG02965', 'HG03139', 'NA19443', 'NA19317', 'NA19347', 'NA20346', 'HG02145', 'HG02258', 'HG02953', 'HG03130', 'HG03521', 'HG02554', 'NA18879', 'HG03209', 'NA18522', 'NA19700', 'HG02055', 'NA19705', 'NA18612', 'NA18620', 'HG04157', 'NA18989', 'NA18971', 'NA18974', 'HG02040', 'NA20870', 'HG03710', 'HG01099', 'HG03579', 'NA21093', 'HG04187', 'HG03009', 'HG03942', 'HG01167', 'HG00140', 'HG00321', 'NA20905', 'HG02492', 'HG02735', 'NA20805', 'NA20809', 'HG01255', 'HG01258', 'HG01433', 'HG003', 'HG002', 'HG03688', 'HG04199', 'HG01192', 'HG01530', 'HG03742', 'NA18608', 'NA18747', 'HG00280', 'HG00329', 'HG00290', 'HG00358', 'HG02015', 'HG02083', 'HG02514', 'HG02074', 'NA18534', 'HG02027', 'HG0

In [29]:
blockDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/DavidPorubsky/AZFc_ColorBLock_Clusters_12162025.csv').drop(columns=['Unnamed: 0'])

In [30]:
colorGroupList=[]
for row in blockDF.index:
    seq1 = blockDF.at[row,'colorblock']
    if 'IR1' in seq1:
        colorGroupList.append('IR1')
    elif 'spacer' in seq1:
        colorGroupList.append('spacer')
    elif 'plus' in seq1:
        colorGroupList.append('plus')
    else:
        colorGroupList.append(seq1[:-1])

blockDF['MainColors']=colorGroupList

In [17]:
otherColors={'spacer','plus','IR1'}

In [18]:
testColors = {'blue', 'gray', 'green', 'red', 'teal', 'yellow'}

In [19]:
testColors2 = {'blue', 'gray', 'green', 'red', 'teal', 'yellow','spacer','plus','IR1'}

In [20]:
b2b3Inversions=[
    ['HG01109','NA19331'],
    ['HG01074','NA19331'],
    ['NA21093','HG03009'],
    ['NA20870','HG02040'],
    ['HG02074','HG02392']
]

In [21]:
myt2tvshg38stylePairings=[
    ['HG01952','HG01928'],
    ['HG02071','HG00706'],
    ['HG02717','HG03471',],
    ['NA18620','HG02040'],
    ['HG04199','HG01192'],
    ['HG02492','HG01167'],
    ['HG03688','HG01192'],
    ['HG02514','HG02392'],
    ['HG02027','HG02392'],
    ['HG02083','HG02392'],
    ['HG00642','200085'],
    ['HG002','HG01167'],
    ['HG01255','HG01167'],
    ['HG01258','HG01167'],
]

In [22]:
pairingList = b2b3Inversions+myt2tvshg38stylePairings

In [23]:
assemblyDict={}
directory='/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
for file in os.listdir(directory):
    if '.fai' in file or '.gzi' in file or '.DS' in file:
        continue
    else:
        assemblyDict[file.split("_")[0]]= directory+file
print(len(assemblyDict))

143


In [24]:
import os
import pysam

base_out = '/LeeLab/HPRC/chromosomeY/Data/colorBlock_Mappings/breakpointpairings/'
pairing_txt = '/LeeLab/HPRC/chromosomeY/Data/colorBlock_Mappings/breakpointpairingFileList.txt'

made_fastas = set()

with open(pairing_txt, 'a+') as outFile:
    for pairing in pairingList:
        sampleOne = pairing[0]
        sampleTwo = pairing[1]

        for color in testColors2:

            sampleOneDF = blockDF[(blockDF['sample'] == sampleOne) & (blockDF['MainColors'] == color)].copy()
            sampleTwoDF = blockDF[(blockDF['sample'] == sampleTwo) & (blockDF['MainColors'] == color)].copy()

            countList = {}

            count = 1
            for row in sampleTwoDF.index:
                fname = f"{sampleTwo}-{count}-{color}.fasta"
                fpath = os.path.join(base_out, fname)

                coordinateName = (
                    f"{sampleTwoDF.at[row, 'contig']}:{sampleTwoDF.at[row, 'start']}-"
                    f"{sampleTwoDF.at[row, 'end']}"
                )


                countList[fname] = coordinateName

                if fname not in made_fastas:
                    with open(fpath, 'w') as fh:
                        fh.write(pysam.faidx(assemblyDict[sampleTwo], coordinateName))
                    made_fastas.add(fname)

                count += 1

            count = 1
            for row in sampleOneDF.index:
                fname = f"{sampleOne}-{count}-{color}.fasta"
                fpath = os.path.join(base_out, fname)

                coordinateName = (
                    f"{sampleOneDF.at[row, 'contig']}:{sampleOneDF.at[row, 'start']}-"
                    f"{sampleOneDF.at[row, 'end']}"
                )

                countList[fname] = coordinateName

                if fname not in made_fastas:
                    with open(fpath, 'w') as fh:
                        fh.write(pysam.faidx(assemblyDict[sampleOne], coordinateName))
                    made_fastas.add(fname)

                count += 1

            countList2 = {coord: fname for fname, coord in countList.items()}

            for row in sampleOneDF.index:
                coord1 = f"{sampleOneDF.at[row, 'contig']}:{sampleOneDF.at[row, 'start']}-{sampleOneDF.at[row, 'end']}"
                for row2 in sampleTwoDF.index:
                    coord2 = f"{sampleTwoDF.at[row2, 'contig']}:{sampleTwoDF.at[row2, 'start']}-{sampleTwoDF.at[row2, 'end']}"
                    fasta1 = countList2[coord1]
                    fasta2 = countList2[coord2]
                    outFile.write(
                        f"/projects/ch-lee-lab/USERS/loftum/chrY/ColorBlocks/minimapRuns/breakpointpairings/{fasta1},"
                        f"/projects/ch-lee-lab/USERS/loftum/chrY/ColorBlocks/minimapRuns/breakpointpairings/{fasta2}\n"
                    )
